# IEEE-CIS Fraud Detection — Exploratory Data Analysis

**Purpose:** Understand the dataset before modelling. This notebook covers:
1. Dataset overview (shape, dtypes, memory)
2. Target distribution (class imbalance)
3. Transaction amount analysis
4. Temporal fraud patterns
5. Categorical fraud rates (card type, email domain, device)
6. V-column null rate audit
7. Correlation with isFraud
8. Identity feature coverage

**Data source:** Reads from `data/processed/merged_data.parquet` (created by the ETL step).
If that file doesn't exist yet, run: `python main.py --only ingest etl`

**All plots are saved to `visuals/` as PNG files for inclusion in the README.**

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path so src.db is importable
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

VISUALS_DIR   = PROJECT_ROOT / 'visuals'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
VISUALS_DIR.mkdir(exist_ok=True)

# Plot style
plt.rcParams.update({
    'figure.dpi':      120,
    'figure.figsize':  (10, 5),
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'font.size':         11,
})
PALETTE = {'legit': '#4C72B0', 'fraud': '#DD8452'}

print('Libraries loaded.')

## 1. Load Data

In [ ]:
parquet_path = PROCESSED_DIR / 'merged_data.parquet'

if parquet_path.exists():
    print(f'Loading from parquet: {parquet_path}')
    # Read a representative subset of columns to avoid loading all 450 cols
    # For EDA we focus on key columns; V cols loaded separately for null audit
    KEY_COLS = [
        'TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'ProductCD',
        'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
        'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain', 'R_emaildomain',
        'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10',
        'C11', 'C12', 'C13', 'C14',
        'D1', 'D2', 'D3', 'D4', 'D5', 'D10', 'D11', 'D15',
        'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9',
        'DeviceType', 'DeviceInfo', 'id_01', 'id_02', 'id_12', 'id_15', 'id_30', 'id_31',
    ]
    df = pd.read_parquet(parquet_path, columns=KEY_COLS)
else:
    # Fallback: load from PostgreSQL
    print('Parquet not found. Loading key columns from PostgreSQL...')
    from src.db import get_engine
    engine = get_engine()
    df = pd.read_sql(
        f'SELECT {', '.join([f\'"' + c + '"\' for c in KEY_COLS])} FROM merged_data LIMIT 200000',
        engine
    )

print(f'Shape: {df.shape}')
print(f'Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
df.head(3)

## 2. Target Distribution

In [ ]:
fraud_counts = df['isFraud'].value_counts().sort_index()
fraud_rate   = df['isFraud'].mean() * 100

print(f"Total transactions: {len(df):,}")
print(f"Fraud:    {fraud_counts[1]:,} ({fraud_rate:.2f}%)")
print(f"Legit:    {fraud_counts[0]:,} ({100-fraud_rate:.2f}%)")
print(f"\nClass imbalance ratio: 1 fraud per {int(1/df['isFraud'].mean()):,} legit transactions")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
colors = [PALETTE['legit'], PALETTE['fraud']]
bars = ax1.bar(['Legitimate', 'Fraud'], fraud_counts.values, color=colors, width=0.5)
for bar, val in zip(bars, fraud_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2000,
             f'{val:,}', ha='center', va='bottom', fontsize=11)
ax1.set_title('Transaction Count by Class', fontsize=13)
ax1.set_ylabel('Count')

# Pie chart
ax2.pie(
    fraud_counts.values,
    labels=[f'Legitimate\n{100-fraud_rate:.1f}%', f'Fraud\n{fraud_rate:.1f}%'],
    colors=colors,
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    textprops={'fontsize': 11},
)
ax2.set_title('Class Distribution', fontsize=13)

plt.suptitle('IEEE-CIS Fraud Detection — Class Imbalance', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(VISUALS_DIR / 'target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: visuals/target_distribution.png')

## 3. Transaction Amount Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Log-scale histogram by fraud label
for label, color in [(0, PALETTE['legit']), (1, PALETTE['fraud'])]:
    subset = df[df['isFraud'] == label]['TransactionAmt']
    axes[0].hist(
        np.log1p(subset), bins=60, alpha=0.6,
        color=color, label='Fraud' if label == 1 else 'Legitimate',
        density=True
    )
axes[0].set_xlabel('log(1 + TransactionAmt)')
axes[0].set_ylabel('Density')
axes[0].set_title('Transaction Amount Distribution (log scale)')
axes[0].legend()

# Fraud rate by amount bucket
bins   = [0, 10, 50, 200, 1000, df['TransactionAmt'].max() + 1]
labels = ['<$10', '$10-$50', '$50-$200', '$200-$1000', '>$1000']
df['amt_bucket'] = pd.cut(df['TransactionAmt'], bins=bins, labels=labels, right=False)
bucket_fraud = df.groupby('amt_bucket')['isFraud'].mean() * 100
bucket_fraud.plot(kind='bar', ax=axes[1], color=PALETTE['fraud'], edgecolor='white')
axes[1].set_title('Fraud Rate by Amount Range')
axes[1].set_xlabel('Amount Range')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].tick_params(axis='x', rotation=0)
for i, v in enumerate(bucket_fraud):
    axes[1].text(i, v + 0.1, f'{v:.1f}%', ha='center', fontsize=9)

plt.suptitle('Transaction Amount Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(VISUALS_DIR / 'amount_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: visuals/amount_distribution.png')

## 4. Temporal Fraud Patterns

In [ ]:
# Derive time features from TransactionDT (seconds offset)
df['tx_hour'] = (df['TransactionDT'] // 3600) % 24
df['tx_dow']  = (df['TransactionDT'] // 86400) % 7

# Heatmap: fraud rate by hour × day of week
pivot = df.groupby(['tx_dow', 'tx_hour'])['isFraud'].mean().unstack() * 100
day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.heatmap(
    pivot,
    ax=axes[0],
    cmap='YlOrRd',
    fmt='.1f',
    cbar_kws={'label': 'Fraud Rate (%)'},
    yticklabels=day_labels,
)
axes[0].set_title('Fraud Rate Heatmap (Hour × Day of Week)')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Day of Week')

# Fraud rate by hour of day (line chart)
hourly_fraud = df.groupby('tx_hour')['isFraud'].mean() * 100
hourly_count = df.groupby('tx_hour').size()
ax_line = axes[1]
ax_twin = ax_line.twinx()
ax_line.plot(hourly_fraud.index, hourly_fraud.values, color=PALETTE['fraud'], lw=2, marker='o', ms=4, label='Fraud Rate')
ax_twin.bar(hourly_count.index, hourly_count.values, alpha=0.2, color=PALETTE['legit'], label='Volume')
ax_line.set_xlabel('Hour of Day')
ax_line.set_ylabel('Fraud Rate (%)', color=PALETTE['fraud'])
ax_twin.set_ylabel('Transaction Volume', color=PALETTE['legit'])
ax_line.set_title('Fraud Rate by Hour of Day')
ax_line.set_xticks(range(0, 24, 2))

plt.suptitle('Temporal Fraud Patterns', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(VISUALS_DIR / 'temporal_patterns.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: visuals/temporal_patterns.png')

## 5. Categorical Fraud Rates

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

def plot_fraud_by_cat(ax, col, title, n_top=None, horizontal=False):
    data = df.groupby(col)['isFraud'].agg(['mean', 'count'])
    data['fraud_rate_pct'] = data['mean'] * 100
    data = data.sort_values('fraud_rate_pct', ascending=False)
    if n_top:
        data = data.head(n_top)
    if horizontal:
        data['fraud_rate_pct'].plot(kind='barh', ax=ax, color=PALETTE['fraud'], edgecolor='white')
        ax.set_xlabel('Fraud Rate (%)')
        for i, (idx, row) in enumerate(data.iterrows()):
            ax.text(row['fraud_rate_pct'] + 0.05, i, f"{row['fraud_rate_pct']:.1f}%", va='center', fontsize=8)
    else:
        data['fraud_rate_pct'].plot(kind='bar', ax=ax, color=PALETTE['fraud'], edgecolor='white')
        ax.set_ylabel('Fraud Rate (%)')
        ax.tick_params(axis='x', rotation=30)
        for i, v in enumerate(data['fraud_rate_pct']):
            ax.text(i, v + 0.1, f'{v:.1f}%', ha='center', fontsize=8)
    ax.set_title(title)

plot_fraud_by_cat(axes[0, 0], 'card4', 'Fraud Rate by Card Network')
plot_fraud_by_cat(axes[0, 1], 'card6', 'Fraud Rate by Card Type (Credit/Debit)')
plot_fraud_by_cat(axes[1, 0], 'P_emaildomain', 'Fraud Rate by Purchaser Email Domain (Top 15)',
                  n_top=15, horizontal=True)
plot_fraud_by_cat(axes[1, 1], 'DeviceType', 'Fraud Rate by Device Type')

plt.suptitle('Categorical Fraud Rates', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(VISUALS_DIR / 'categorical_fraud_rates.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: visuals/categorical_fraud_rates.png')

## 6. V-Column Null Rate Audit

In [ ]:
null_rates_path = PROCESSED_DIR / 'v_null_rates.csv'

if null_rates_path.exists():
    null_df = pd.read_csv(null_rates_path)
    v_null  = null_df[null_df['col'].str.startswith('V')].copy()
    v_null  = v_null.sort_values('null_pct', ascending=False).reset_index(drop=True)
    
    THRESHOLD = 80.0
    drop_count = (v_null['null_pct'] > THRESHOLD).sum()
    keep_count = len(v_null) - drop_count
    
    print(f"V columns: {len(v_null)} total")
    print(f"  Will be DROPPED (>{THRESHOLD:.0f}% null): {drop_count}")
    print(f"  Will be KEPT   (<={THRESHOLD:.0f}% null): {keep_count}")

    fig, ax = plt.subplots(figsize=(16, 5))
    colors = ['#DD8452' if n > THRESHOLD else '#4C72B0' for n in v_null['null_pct']]
    ax.bar(v_null.index, v_null['null_pct'], color=colors, width=1.0, edgecolor='none')
    ax.axhline(y=THRESHOLD, color='red', linestyle='--', lw=1.5, label=f'{THRESHOLD:.0f}% drop threshold')
    ax.set_xlabel('V Column (sorted by null %)')
    ax.set_ylabel('Null Rate (%)')
    ax.set_title(f'V-Column Null Rates — {drop_count} dropped (orange), {keep_count} kept (blue)')
    ax.legend()
    ax.set_xticks([])
    plt.tight_layout()
    plt.savefig(VISUALS_DIR / 'v_column_nulls.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: visuals/v_column_nulls.png')
else:
    print(f'v_null_rates.csv not found at {null_rates_path}')
    print('Run ETL step first: python main.py --only etl')

## 7. Correlation with isFraud

In [ ]:
# Compute Pearson correlation of numeric columns with isFraud
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['TransactionID', 'isFraud', 'TransactionDT',
                                                       'tx_hour', 'tx_dow', 'amt_bucket']]

corr = df[numeric_cols + ['isFraud']].corr()['isFraud'].drop('isFraud')
top20_pos = corr.nlargest(10)
top20_neg = corr.nsmallest(10)
top20 = pd.concat([top20_pos, top20_neg]).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#DD8452' if v > 0 else '#4C72B0' for v in top20.values]
top20.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson Correlation with isFraud')
ax.set_title('Top 20 Feature Correlations with Fraud Label')
plt.tight_layout()
plt.savefig(VISUALS_DIR / 'correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: visuals/correlation_heatmap.png')

print('\nTop 10 positively correlated features:')
print(top20_pos.to_string())
print('\nTop 10 negatively correlated features:')
print(top20_neg.to_string())

## 8. Identity Feature Coverage

In [ ]:
# What % of transactions have identity data?
has_identity = df['DeviceType'].notna()
coverage_pct = has_identity.mean() * 100
print(f"Transactions with identity data: {has_identity.sum():,} ({coverage_pct:.1f}%)")
print(f"Transactions without identity:   {(~has_identity).sum():,} ({100-coverage_pct:.1f}%)")

# Fraud rate with vs without identity
fraud_with_id    = df[has_identity]['isFraud'].mean() * 100
fraud_without_id = df[~has_identity]['isFraud'].mean() * 100
print(f"\nFraud rate WITH identity:    {fraud_with_id:.2f}%")
print(f"Fraud rate WITHOUT identity: {fraud_without_id:.2f}%")

# DeviceType fraud rate
print("\nFraud rate by DeviceType:")
device_fraud = df.groupby('DeviceType')['isFraud'].agg(['mean', 'count'])
device_fraud['fraud_rate_pct'] = device_fraud['mean'] * 100
print(device_fraud[['count', 'fraud_rate_pct']].sort_values('fraud_rate_pct', ascending=False))

# Top 15 DeviceInfo values by fraud rate (min 50 transactions)
device_info_stats = (
    df.groupby('DeviceInfo')['isFraud']
    .agg(['mean', 'count'])
    .query('count >= 50')
    .sort_values('mean', ascending=False)
    .head(15)
)
device_info_stats['fraud_rate_pct'] = device_info_stats['mean'] * 100

fig, ax = plt.subplots(figsize=(10, 6))
device_info_stats['fraud_rate_pct'].plot(
    kind='barh', ax=ax, color=PALETTE['fraud'], edgecolor='white'
)
ax.set_xlabel('Fraud Rate (%)')
ax.set_title('Top 15 DeviceInfo Values by Fraud Rate (min 50 transactions)')
plt.tight_layout()
plt.savefig(VISUALS_DIR / 'device_info_fraud_rates.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: visuals/device_info_fraud_rates.png')

## Summary

Key findings from EDA:

| Finding | Value |
|---|---|
| Overall fraud rate | ~3.5% |
| Identity coverage | ~24% of transactions |
| V columns to drop | See `v_column_nulls.png` |
| Strongest numeric correlations | See `correlation_heatmap.png` |

**Next step:** `python main.py --skip ingest etl` to run feature engineering and training.

All visualisations are saved in `../visuals/` and referenced in the README.